# Persian TeCo — verify inputs before word ERPs
Use **CPU → Runtime → Run all** and allow Drive. This inspects the existing `Thesis/Data/TRT_Total` feature files and one existing raw `teco_pickles/*/task1.pickle` file. No dataset download or EEG extraction occurs. The raw pickle may be about 2 GB and is read in Colab memory only. Persian text and reports stay in Drive.

Outputs: `Thesis/Results/word_sentiment_eeg/teco_preflight/structure.json` and `word_inventory.csv`. This is a preflight, not a TeCo ERP result. Word timing, actual sampling rate, montage and Persian labels must be verified before epoching.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys
drive.mount('/content/drive')
THESIS = Path('/content/drive/MyDrive/Thesis')
REPO = Path('/content/word-sentiment-eeg-teco')
def git(*args):
    subprocess.run(['git', '-c', 'credential.helper=', '-c', 'http.extraheader=', *map(str,args)], check=True, env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone', '--depth', '1', 'https://github.com/parmisbathayan/word-sentiment-eeg.git', REPO)
else:
    if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Temporary TeCo checkout has edits. Save them before updating.')
    git('-C', REPO, 'pull', '--ff-only', 'origin', 'main')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas'], check=True)
print('Code version:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
sys.path.insert(0,str(REPO / 'scripts'))
from colab_progress import run_with_progress
command = [sys.executable, '-u', str(REPO / 'scripts/teco_preflight.py'), '--thesis-root', str(THESIS), '--inspect-raw']
run_with_progress(command, THESIS / 'Results/word_sentiment_eeg/teco_preflight/progress.json', activity='Persian TeCo input preflight')
